# 6.9 讀檔分段會改變切詞嗎？

# 第 6 章：自己的中英文 tokenizer

前置：字元ID與下一字loss。Tokenizer訓練是統計切詞規則，不是訓練LLM。早期與CLI使用可還原ByteTokenizer；BPE在本章獨立學習，換模型時要同步詞表與checkpoint。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：常見卡片合併：不是先懂詞義**

每次合併一對常見相鄰符號。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/bpe.svg")))

**先想一想：**切在常見詞中間，token序列可能變嗎？

同一文件被I/O切成chunks不代表語意上有邊界。若每chunk獨立BPE，跨chunk的merge消失；若raw bytes切半個字，提前decode還會出錯。

**把直覺寫成數學：**一般encode(a+b)不一定等於encode(a)+encode(b)。不同文件則應先明確EOS與邊界契約。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tokenizers import Tokenizer, models, pre_tokenizers, trainers

tok = Tokenizer(models.BPE(unk_token="[UNK]"))
tok.pre_tokenizer = pre_tokenizers.Whitespace()
tok.train_from_iterator(["hello hello hello"], trainers.BpeTrainer(vocab_size=20, special_tokens=["[UNK]"]))
whole = tok.encode("hello").ids
chunks = tok.encode("hel").ids + tok.encode("lo").ids
print(whole, chunks)
assert whole != chunks

**如何讀結果：**這個反例是文字切分，不是圖像patchify；生產讀檔需留跨chunk尾段或按文件encode。

**只改一件事：**只把chunk邊界改到詞與詞之間。
